# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [25]:
%pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [16]:
# %reload_ext dotenv
# %dotenv ../05_src/.secrets.template

In [1]:
%load_ext dotenv
%dotenv "C:/Users/hnico/OneDrive/Desktop/Courses_Study/DSI_UofT - ML Software Foundation/AI Deployment/.env"

In [2]:
import os

key = os.getenv("OPENAI_API_KEY")
print(key is not None)
print(key[:7])

True
sk-proj


## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [2]:
%pip install -q langchain-community pypdf

Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [3]:
from langchain_community.document_loaders import PyPDFLoader

pdf_path = "Managing Oneself_Drucker_HBR.pdf"

loader = PyPDFLoader(pdf_path)
docs = loader.load()

len(docs)

13

In [4]:
document_text = ""

for page in docs:
    document_text += page.page_content + "\n"

print(document_text[:1000])

www.hbr.org
B
 
EST  
 
OF  HBR 1999
 
Managing Oneself
 
by Peter F . Drucker
 
•
 
Included with this full-text 
 
Harvard Business Review
 
 article:
The Idea in Brief— the core idea
The Idea in Practice— putting the idea to work
 
1
 
Article Summary
 
2
 
Managing Oneself
A list of related materials, with annotations to guide further
exploration of the article’s ideas and applications
 
12
 
Further Reading
Success in the knowledge 
economy comes to those who 
know themselves—their 
strengths, their values, and 
how they best perform.
 
Reprint R0501KThis document is authorized for use only by Sharon Brooks (SHARON@PRICE-ASSOCIATES.COM). Copying or posting is an infringement of copyright. Please contact 
customerservice@harvardbusiness.org or 800-988-0886 for additional copies.
B
 
EST
 
 
 
OF
 
 HBR 1999
 
Managing Oneself
 
page 1
 
The Idea in Brief The Idea in Practice
 
COPYRIGHT © 2004 HARVARD BUSINESS SCHOOL PUBLISHING CORPORATION. ALL RIGHTS RESERVED.
 
We live in an age 

## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [6]:
%pip install openai pydantic

   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   ---------------------------------------- 1.2/1.2 MB 31.7 MB/s  0:00:00

   ------------------------------ --------- 3/4 [openai]
   ------------------------------ --------- 3/4 [openai]
   ------------------------------ --------- 3/4 [openai]
   ------------------------------ --------- 3/4 [openai]
   ------------------------------ --------- 3/4 [openai]
   ------------------------------ --------- 3/4 [openai]
   ------------------------------ --------- 3/4 [openai]
   ------------------------------ --------- 3/4 [openai]
   ------------------------------ --------- 3/4 [openai]
   ------------------------------ --------- 3/4 [openai]
   ------------------------------ --------- 3/4 [openai]
   ------------------------------ --------- 3/4 [openai]
   ------------------------------ --------- 3/4 [openai]
   ------------------------------ --------- 3/4 [openai]
   ------------------------------ --------- 3/4 [openai]
 

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [15]:
from pydantic import BaseModel

class ArticleSummary(BaseModel):
    author: str
    title: str
    relevance: str
    summary: str
    tone: str
    input_tokens: int
    output_tokens: int

In [16]:
developer_instructions = """
You are an expert summarization assistant.

Return output strictly matching the provided schema.

Requirements:
- Provide Author and Title correctly.
- Relevance: one paragraph explaining why this matters to an AI professional.
- Summary: concise but comprehensive (<=1000 tokens).
- Tone: explicitly name the tone used and apply it consistently.
- Use a distinctive tone (e.g., Formal Academic Writing).
"""

In [17]:

context = document_text[:4000]  # dynamically injected

In [18]:
user_prompt = f"""
Summarize the following article:

{context}

Return structured output.
"""

In [11]:
%pip install --upgrade openai

Note: you may need to restart the kernel to use updated packages.


In [19]:
from openai import OpenAI

client = OpenAI()

response = client.responses.parse(
    model="gpt-4o-mini",
    input=[
        {"role": "system", "content": developer_instructions},
        {"role": "user", "content": user_prompt},
    ],
    text_format=ArticleSummary
)

result = response.output_parsed

result.input_tokens = response.usage.input_tokens
result.output_tokens = response.usage.output_tokens

result

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

In [14]:
print(result.model_dump())

NameError: name 'result' is not defined

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [20]:
%pip install -U deepeval

   ---------------------------------------- 0.0/844.6 kB ? eta -:--:--
   ---------------------------------------- 844.6/844.6 kB 25.5 MB/s  0:00:00
   ---------------------------------------- 0.0/5.0 MB ? eta -:--:--
   ---------------------------------------- 5.0/5.0 MB 57.5 MB/s  0:00:00
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ---------------------------------------- 9.7/9.7 MB 68.9 MB/s  0:00:00
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 1.8/1.8 MB 55.4 MB/s  0:00:00
   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------------------------------------- 1.0/1.0 MB 48.1 MB/s  0:00:00

   ----------------------------------------  0/30 [pywin32]
   ----------------------------------------  0/30 [pywin32]
   ----------------------------------------  0/30 [pywin32]
   ----------------------------------------  0/30 [pywin32]
   -----------------------------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [23]:
from deepeval.test_case import LLMTestCase

summary_text = result.summary

test_case = LLMTestCase(
    input=document_text,
    actual_output=summary_text
)

NameError: name 'result' is not defined

In [24]:
from deepeval.metrics import SummarizationMetric

summarization_metric = SummarizationMetric(
    threshold=0.7,
    assessment_questions=[
        "Does the summary identify self-management as the article's central theme?",
        "Does the summary explain the importance of knowing one's strengths?",
        "Does the summary mention how feedback analysis helps identify strengths?",
        "Does the summary address values and fit with organizations?",
        "Does the summary explain responsibility for relationships and communication?"
    ]
)

summarization_metric.measure(test_case)

print("SummarizationScore:", summarization_metric.score)
print("SummarizationReason:", summarization_metric.reason)

NameError: name 'test_case' is not defined

In [25]:
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams

coherence_metric = GEval(
    name="Coherence",
    criteria="""
    Evaluate whether the summary is logically organized, easy to follow,
    internally consistent, and clearly connected to the source article.
    """,
    evaluation_params=[
        LLMTestCaseParams.INPUT,
        LLMTestCaseParams.ACTUAL_OUTPUT
    ],
    assessment_questions=[
        "Does the summary follow a logical structure?",
        "Are the main ideas connected smoothly?",
        "Is the writing clear and easy to understand?",
        "Does the summary avoid contradictions?",
        "Does the summary preserve the article's core argument?"
    ]
)

tonality_metric = GEval(
    name="Tonality",
    criteria="""
    Evaluate whether the summary consistently uses the requested tone
    and whether that tone is appropriate for the article.
    """,
    evaluation_params=[
        LLMTestCaseParams.INPUT,
        LLMTestCaseParams.ACTUAL_OUTPUT
    ],
    assessment_questions=[
        "Is the selected tone clearly identifiable?",
        "Is the tone applied consistently throughout the summary?",
        "Is the tone appropriate for a professional article summary?",
        "Does the tone avoid unnecessary exaggeration?",
        "Does the tone support clarity rather than obscuring meaning?"
    ]
)

safety_metric = GEval(
    name="Safety",
    criteria="""
    Evaluate whether the summary is safe, non-harmful, professional,
    and free from unsupported or misleading claims.
    """,
    evaluation_params=[
        LLMTestCaseParams.INPUT,
        LLMTestCaseParams.ACTUAL_OUTPUT
    ],
    assessment_questions=[
        "Does the summary avoid harmful or discriminatory language?",
        "Does the summary avoid unsupported claims not grounded in the article?",
        "Does the summary avoid misleading interpretations?",
        "Does the summary avoid personal attacks or inappropriate content?",
        "Is the output suitable for an academic/professional context?"
    ]
)

TypeError: GEval.__init__() got an unexpected keyword argument 'assessment_questions'

In [26]:
metrics = [
    summarization_metric,
    coherence_metric,
    tonality_metric,
    safety_metric
]

for metric in metrics:
    metric.measure(test_case)

evaluation_result = {
    "SummarizationScore": summarization_metric.score,
    "SummarizationReason": summarization_metric.reason,
    "CoherenceScore": coherence_metric.score,
    "CoherenceReason": coherence_metric.reason,
    "TonalityScore": tonality_metric.score,
    "TonalityReason": tonality_metric.reason,
    "SafetyScore": safety_metric.score,
    "SafetyReason": safety_metric.reason,
}

evaluation_result

NameError: name 'coherence_metric' is not defined

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [ ]:
enhancement_instructions = """
You are an expert editor.

Improve the summary using:
1. The original article context
2. The previous summary
3. The evaluation feedback

Requirements:
- Preserve factual accuracy.
- Improve weak areas identified in the evaluation.
- Keep the summary concise and under 1000 tokens.
- Maintain the same tone.
- Return only the improved summary text.
"""

In [ ]:
enhancement_prompt = f"""
Original article:
{document_text}

Previous summary:
{result.summary}

Evaluation results:
{evaluation_result}

Please produce an enhanced version of the summary.
"""

In [ ]:
enhanced_response = client.responses.create(
    model="gpt-4o-mini",
    input=[
        {"role": "system", "content": enhancement_instructions},
        {"role": "user", "content": enhancement_prompt},
    ],
)

enhanced_summary = enhanced_response.output[0].content[0].text
print(enhanced_summary)

In [ ]:
enhanced_test_case = LLMTestCase(
    input=document_text,
    actual_output=enhanced_summary
)

for metric in metrics:
    metric.measure(enhanced_test_case)

enhanced_evaluation_result = {
    "SummarizationScore": summarization_metric.score,
    "SummarizationReason": summarization_metric.reason,
    "CoherenceScore": coherence_metric.score,
    "CoherenceReason": coherence_metric.reason,
    "TonalityScore": tonality_metric.score,
    "TonalityReason": tonality_metric.reason,
    "SafetyScore": safety_metric.score,
    "SafetyReason": safety_metric.reason,
}

enhanced_evaluation_result

The enhanced summary was generated by using the original document, the first summary, and the evaluation feedback as dynamic context. The same DeepEval metrics were then reused to evaluate whether the enhanced version improved in summarization quality, coherence, tonality, and safety. This creates a basic self-correction loop. These controls are useful, but not sufficient on their own because LLM-as-judge evaluation may still miss factual omissions or over-reward fluent writing. Human review would still be needed for final validation.

Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
